In [ ]:
import pandas as pd
from sklearn.preprocessing import QuantileTransformer
from torch.utils.data import Dataset, Subset
import torch
import numpy as np
import os
from functools import lru_cache
from tqdm import tqdm
import sys
sys.path.append('../')
import karman
#training
import torch
from torch.utils.data import RandomSampler, SequentialSampler

torch_type=torch.float32
torch.set_default_dtype(torch_type)

In [ ]:
karman_dataset=karman.KarmanDataset(thermo_path='../data/satellites_data_w_sw_2mln.csv',
                            min_date=pd.to_datetime("2000-07-29 01:08:37"),
                            max_date=pd.to_datetime("2024-04-30 23:24:00"),
                            normalization_dict=None,
                            #omni_path='omniweb_1min_data_2001_2022.h5',
                           )

In [ ]:
input_dimension=karman_dataset[0]['instantaneous_features'].shape[0]
device=torch.device('cpu')
karman_model=karman.SimpleNetwork( input_dim=input_dimension,
                            act=torch.nn.LeakyReLU(negative_slope=0.01),
                            hidden_layer_dims=[256, 256, 256],
                            output_dim=1).to(device)

In [ ]:
num_params=sum(p.numel() for p in karman_model.parameters() if p.requires_grad)
print(f'Karman model num parameters: {num_params}')

In [ ]:
#Train, validation, test splits:
idx_test_fold=2
test_month_idx = 2 * (idx_test_fold - 1)
validation_month_idx = test_month_idx + 2
print(test_month_idx,validation_month_idx)
karman_dataset._set_indices(test_month_idx=[test_month_idx], validation_month_idx=[validation_month_idx])
train_dataset = karman_dataset.train_dataset()
validation_dataset = karman_dataset.validation_dataset()
test_dataset = karman_dataset.test_dataset()


print(f'Training dataset example: {train_dataset[0].items()}')

In [ ]:
train_sampler = RandomSampler(train_dataset, num_samples=len(train_dataset))
validation_sampler = SequentialSampler(validation_dataset)
test_sampler = SequentialSampler(test_dataset)

In [ ]:
####### Training Parameters ##########
batch_size = 512
model_path = None #pass a path to a model in case you want to continue training from a file
lr = 0.001
epochs = 100
num_workers=0

In [ ]:
# Here we set the optimizer
optimizer = torch.optim.Adam(karman_model.parameters(), lr=lr,amsgrad=True)
scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[25,50,75,100,125,150,175,200,225,230,240,250,260,270], gamma=0.8, verbose=False)
criterion=torch.nn.MSELoss()

# And the dataloader
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    pin_memory=True,
    num_workers=num_workers,
    sampler=train_sampler,
    drop_last=True,
)
validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=batch_size,
    pin_memory=False,
    num_workers=num_workers,
    sampler=validation_sampler,
    drop_last=False,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    pin_memory=False,
    num_workers=num_workers,
    sampler=test_sampler,
    drop_last=False,
)

In [ ]:
def mean_absolute_percentage_error(y_pred,y_true):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
losses_per_minibatch={'nn_mse_train':[],'nrlmsise00_mse_train':[],'nn_mape_train':[],'nrlmsise00_mape_train':[],
                      'nn_mse_valid':[],'nrlmsise00_mse_valid':[],'nn_mape_valid':[],'nrlmsise00_mape_valid':[]}
losses_total={'nn_mse_train':[],'nrlmsise00_mse_train':[],'nn_mape_train':[],'nrlmsise00_mape_train':[],
              'nn_mse_valid':[],'nrlmsise00_mse_valid':[],'nn_mape_valid':[],'nrlmsise00_mape_valid':[]}

# Training loop

best_loss_total = np.inf
best_loss = np.inf
for epoch in range(epochs):
    #first training loop:
    loss_total_nn=0.
    loss_total_nrlmsise00=0.
    mape_total_nn=0.
    mape_total_nrlmsise00=0.
    #we set the model in training mode:
    karman_model.train()
    for batch_idx,el in enumerate(train_loader):
        minibatch=el['instantaneous_features'].to(device)
        #let's store the normalized and unnormalized target density:
        target=el['target'].to(device)
        rho_target=el['ground_truth'].detach().cpu().numpy()
        #now the normalized and unnormalized NN-predicted density:
        target_nn=karman_model(minibatch).squeeze()
        rho_nn=karman_dataset.unscale_density(target_nn).detach().cpu().numpy()
        #finally the NRLMSISE-00 ones:
        rho_nrlmsise00=el['nrlmsise00'].detach().cpu().numpy()
        target_nrlmsise00=karman_dataset.scale_density(el['nrlmsise00'].to(device))
        #now the loss computation:
        loss_nn = criterion(target_nn, target)
        loss_nrlmsise00 = criterion(target_nrlmsise00, target)

        # Zeroes the gradient 
        optimizer.zero_grad()

        # Backward pass: compute gradient of the loss with respect to model parameters
        loss_nn.backward()

        # Calling the step function on an Optimizer makes an update to its parameters
        optimizer.step()

        #We compute the logged quantities
        losses_per_minibatch['nn_mse_train'].append(loss_nn.item())
        losses_per_minibatch['nrlmsise00_mse_train'].append(loss_nrlmsise00.item())
        losses_per_minibatch['nn_mape_train'].append(mean_absolute_percentage_error(rho_nn, rho_target))
        losses_per_minibatch['nrlmsise00_mape_train'].append(mean_absolute_percentage_error(rho_nrlmsise00, rho_target))
        #now let's also accumulate them for the overall loss computation in each epoch:
        loss_total_nn+=losses_per_minibatch['nn_mse_train'][-1]
        loss_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mse_train'][-1]
        mape_total_nn+=losses_per_minibatch['nn_mape_train'][-1]
        mape_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mape_train'][-1]
        
        #Save the best model (this is wrong and should be done on the dataset):
        if loss_nn.item()<best_loss:    
            best_loss=loss_nn.item()

        #Print every 10 minibatches:
        if batch_idx%10:    
            print(f'minibatch: {batch_idx}/{len(train_loader)}, best minibatch loss till now: {best_loss:.4e}, NN MSE: {losses_per_minibatch['nn_mse_train'][-1]:.10f}, nrlmsise00 MSE: {losses_per_minibatch['nrlmsise00_mse_train'][-1]:.10f}, NN MAPE: {losses_per_minibatch['nn_mape_train'][-1]:.3f}, nrlmsise00 MAPE: {losses_per_minibatch['nrlmsise00_mape_train'][-1]:.3f}', end='\r')
    
    # over the whole dataset, we take the average of the minibatch losses:
    losses_total['nn_mse_train'].append(loss_total_nn/len(train_loader))
    losses_total['nrlmsise00_mse_train'].append(loss_total_nrlmsise00/len(train_loader))
    losses_total['nn_mape_train'].append(mape_total_nn/len(train_loader))
    losses_total['nrlmsise00_mape_train'].append(mape_total_nrlmsise00/len(train_loader))

    #Print at the end of the epoch
    curr_lr = scheduler.optimizer.param_groups[0]['lr']
    print(" "*300, end="\r")    
    print("\nTraining")
    print(f'Epoch {epoch + 1}/{epochs}, lr: {curr_lr}, NN MSE (total): {losses_total['nn_mse_train'][-1]:.7f}, nrlmsise00 MSE (total): {losses_total['nrlmsise00_mse_train'][-1]:.7f}, NN MAPE (total): {losses_total['nn_mape_train'][-1]:.3f}, nrlmsise00 MAPE (total): {losses_total['nrlmsise00_mape_train'][-1]:.3f}')
    # Perform a step in LR scheduler to update LR
    scheduler.step()
    
    #Validation loop:
    loss_total_nn=0.
    loss_total_nrlmsise00=0.
    mape_total_nn=0.
    mape_total_nrlmsise00=0.
    #let's switch the model to evaluation mode:
    karman_model.eval()
    with torch.no_grad():
        for batch_idx,el in enumerate(validation_loader):
            minibatch=el['instantaneous_features'].to(device)
            #let's store the normalized and unnormalized target density:
            target=el['target'].to(device)
            rho_target=el['ground_truth'].detach().cpu().numpy()
            #now the normalized and unnormalized NN-predicted density:
            target_nn=karman_model(minibatch).squeeze()
            rho_nn=karman_dataset.unscale_density(target_nn).detach().cpu().numpy()
            #finally the NRLMSISE-00 ones:
            rho_nrlmsise00=el['nrlmsise00'].detach().cpu().numpy()
            target_nrlmsise00=karman_dataset.scale_density(el['nrlmsise00'].to(device))
            #now the loss computation:
            loss_nn = criterion(target_nn, target)
            loss_nrlmsise00 = criterion(target_nrlmsise00, target)
            #We compute the logged quantities
            losses_per_minibatch['nn_mse_valid'].append(loss_nn.item())
            losses_per_minibatch['nrlmsise00_mse_valid'].append(loss_nrlmsise00.item())
            losses_per_minibatch['nn_mape_valid'].append(mean_absolute_percentage_error(rho_nn, rho_target))
            losses_per_minibatch['nrlmsise00_mape_valid'].append(mean_absolute_percentage_error(rho_nrlmsise00, rho_target))
            #now let's also accumulate them for the overall loss computation in each epoch:
            loss_total_nn+=losses_per_minibatch['nn_mse_valid'][-1]
            loss_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mse_valid'][-1]
            mape_total_nn+=losses_per_minibatch['nn_mape_valid'][-1]
            mape_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mape_valid'][-1]
        # over the whole dataset, we take the average of the minibatch losses:
        losses_total['nn_mse_valid'].append(loss_total_nn/len(validation_loader))
        losses_total['nrlmsise00_mse_valid'].append(loss_total_nrlmsise00/len(validation_loader))
        losses_total['nn_mape_valid'].append(mape_total_nn/len(validation_loader))
        losses_total['nrlmsise00_mape_valid'].append(mape_total_nrlmsise00/len(validation_loader))
    print("\nValidation")
    print(f'Epoch {epoch + 1}/{epochs}, lr: {curr_lr}, NN MSE (total): {losses_total['nn_mse_valid'][-1]:.7f}, nrlmsise00 MSE (total): {losses_total['nrlmsise00_mse_valid'][-1]:.7f}, NN MAPE (total): {losses_total['nn_mape_valid'][-1]:.3f}, nrlmsise00 MAPE (total): {losses_total['nrlmsise00_mape_valid'][-1]:.3f}')

    #updating torch best model:
    if losses_total['nn_mse_valid'][-1] < best_loss_total:
        torch.save(karman_model.state_dict(), f'../models/karman_model_valid_loss_{losses_total['nn_mse_valid'][-1]}.torch')
        best_loss_total=losses_total['nn_mse_valid'][-1]